Linear Algebra 2 - Advanced Tensor Operations and Spectral Theory

Section 17: Advanced Tensor Operations
Einstein summation notation, outer products, Kronecker product,
batch matrix operations, and 4D tensors for CNNs.

Section 18: Spectral Theory
Spectral decomposition, matrix functions (square root, exponential),
pseudoinverse, and condition number and numerical stability.

Section 17 - Advanced Tensor Operations

Einstein summation notation (einsum) is a compact and general way to express
tensor contractions. Any matrix multiply, dot product, outer product, trace,
or batch operation can be written with a single einsum string.

einsum notation: each index letter represents one dimension.
Repeated indices are summed over. Output indices appear on the right of ->.

Examples:
- 'ij,jk->ik'   matrix multiply A @ B
- 'bij,bjk->bik' batch matrix multiply
- 'i,j->ij'     outer product
- 'ii->'        trace
- 'ij,ij->ij'   element-wise multiply (Hadamard)

In [ ]:
import numpy as np

A = np.array([[1.0, 2.0], [3.0, 4.0]])
B = np.array([[5.0, 6.0], [7.0, 8.0]])
v = np.array([1.0, 2.0, 3.0])
u = np.array([4.0, 5.0])

# matrix multiply
print('matrix multiply:')
print(np.einsum('ij,jk->ik', A, B))
print(A @ B)  # same result

# dot product
print('\ndot product:')
print(np.einsum('i,i->', v, v))
print(np.dot(v, v))

# outer product: v (3,) and u (2,) -> (3, 2)
print('\nouter product (3,) x (2,) -> (3,2):')
print(np.einsum('i,j->ij', v, u))
print(np.outer(v, u))

# trace (sum of diagonal)
print('\ntrace:')
print(np.einsum('ii->', A))
print(np.trace(A))

# element-wise multiply
print('\nelement-wise multiply:')
print(np.einsum('ij,ij->ij', A, B))
print(A * B)

In [ ]:
import numpy as np

# batch matrix multiply: useful for processing multiple matrices at once
# shape (batch, m, k) @ (batch, k, n) -> (batch, m, n)

batch, m, k, n = 4, 3, 5, 2
A_batch = np.random.randn(batch, m, k)
B_batch = np.random.randn(batch, k, n)

C_einsum = np.einsum('bij,bjk->bik', A_batch, B_batch)
C_matmul = A_batch @ B_batch  # numpy broadcast matmul

print(f'A_batch shape: {A_batch.shape}')
print(f'B_batch shape: {B_batch.shape}')
print(f'C shape      : {C_einsum.shape}')
print(f'einsum == matmul: {np.allclose(C_einsum, C_matmul)}')

# batch dot products: (batch, d) dot each row with itself -> (batch,) norms squared
X = np.random.randn(8, 6)   # 8 vectors in R^6
norms_sq = np.einsum('bi,bi->b', X, X)
print(f'\nnorms squared via einsum: {norms_sq.round(4)}')
print(f'verify: {np.allclose(norms_sq, (X**2).sum(axis=1))}')

In [ ]:
import numpy as np

# 4D tensors: (batch, channels, height, width) -- CNN feature maps

batch, C_in, H, W = 2, 3, 4, 4   # 2 images, 3 channels, 4x4 spatial
X_cnn = np.random.randn(batch, C_in, H, W)

print(f'CNN input shape: {X_cnn.shape}  (batch, channels, height, width)')

# global average pooling: average over spatial dims -> (batch, channels)
X_pooled = X_cnn.mean(axis=(2, 3))
print(f'after global avg pool: {X_pooled.shape}  (batch, channels)')

# channel-wise normalisation (like batch norm)
mean = X_cnn.mean(axis=(0, 2, 3), keepdims=True)  # per channel
std  = X_cnn.std(axis=(0, 2, 3),  keepdims=True)
X_norm = (X_cnn - mean) / (std + 1e-8)
print(f'normalised mean per channel: {X_norm.mean(axis=(0,2,3)).round(6)}')
print(f'normalised std  per channel: {X_norm.std(axis=(0,2,3)).round(4)}')

# einsum for a learned linear combination of channels (1x1 convolution)
C_out = 5
W_conv = np.random.randn(C_out, C_in) * 0.1  # weight matrix
# output[b, c_out, h, w] = sum_c W_conv[c_out, c] * X[b, c, h, w]
X_out = np.einsum('oc,bchw->bohw', W_conv, X_cnn)
print(f'\n1x1 conv output: {X_out.shape}  (batch, C_out, H, W)')

In [ ]:
import torch

# PyTorch einsum covers the same operations
A = torch.randn(3, 4)
B = torch.randn(4, 5)

print('matrix multiply via einsum:')
print(torch.einsum('ij,jk->ik', A, B).shape)

# multi-head attention pattern (simplified)
# Q: (batch, heads, seq, d_k)
# K: (batch, heads, seq, d_k)
# scores[b, h, i, j] = sum_d Q[b,h,i,d] * K[b,h,j,d]
batch, heads, seq, d_k = 2, 4, 8, 16
Q = torch.randn(batch, heads, seq, d_k)
K = torch.randn(batch, heads, seq, d_k)
V = torch.randn(batch, heads, seq, d_k)

scores = torch.einsum('bhid,bhjd->bhij', Q, K) / (d_k ** 0.5)
weights = torch.softmax(scores, dim=-1)
output  = torch.einsum('bhij,bhjd->bhid', weights, V)

print(f'\nattention scores shape : {scores.shape}')
print(f'attention weights shape: {weights.shape}')
print(f'attention output shape : {output.shape}')

In [ ]:
import tensorflow as tf

# TensorFlow einsum
A = tf.random.normal([3, 4])
B = tf.random.normal([4, 5])

C = tf.einsum('ij,jk->ik', A, B)
print(f'matmul via tf.einsum: {C.shape}')

# outer product
v = tf.constant([1.0, 2.0, 3.0])
u = tf.constant([4.0, 5.0])
outer = tf.einsum('i,j->ij', v, u)
print(f'outer product shape: {outer.shape}')
print(f'outer product:\n{outer.numpy()}')

# Kronecker product using einsum: kron(A, B) for small matrices
# kron[ac, bd] = A[a,b] * B[c,d]
A_small = tf.constant([[1.0, 2.0], [3.0, 4.0]])
B_small = tf.constant([[0.0, 5.0], [6.0, 7.0]])
kron = tf.einsum('ab,cd->acbd', A_small, B_small)
kron = tf.reshape(kron, [4, 4])
print(f'\nKronecker product (2x2 kron 2x2) -> 4x4:\n{kron.numpy()}')

Section 18 - Spectral Theory

The spectral theorem states: any real symmetric matrix A can be decomposed as
A = Q diag(lambda) Q^T where Q has orthonormal columns and lambda are real eigenvalues.
This is called the spectral decomposition or eigendecomposition.

Matrix functions extend scalar functions to matrices using spectral decomposition:
f(A) = Q diag(f(lambda_1), ..., f(lambda_n)) Q^T
This defines the matrix square root, matrix exponential, matrix logarithm, etc.

The condition number kappa(A) = sigma_max / sigma_min (ratio of largest to smallest
singular value) measures numerical sensitivity. High condition number means small
input perturbations cause large output changes -- the matrix is ill-conditioned.

In [ ]:
# Base Python: power iteration -- finds the dominant eigenvector
import math

def power_iteration(A, num_iters=100):
    n = len(A)
    v = [1.0 / math.sqrt(n)] * n   # start with uniform vector
    for _ in range(num_iters):
        w = [sum(A[i][j] * v[j] for j in range(n)) for i in range(n)]
        norm = math.sqrt(sum(x**2 for x in w))
        v = [x / norm for x in w]
    # Rayleigh quotient: eigenvalue = v^T A v
    Av = [sum(A[i][j] * v[j] for j in range(n)) for i in range(n)]
    lam = sum(v[i] * Av[i] for i in range(n))
    return lam, v

A = [[4.0, 2.0],
     [2.0, 3.0]]

lam, v = power_iteration(A)
print(f'dominant eigenvalue : {lam:.4f}')
print(f'dominant eigenvector: {[round(x, 4) for x in v]}')

# verify using numpy
import numpy as np
eigvals, eigvecs = np.linalg.eigh(np.array(A))
print(f'\nnumpy eigenvalues: {eigvals[::-1].round(4)}  (largest first)')
print(f'numpy eigenvector: {eigvecs[:, -1].round(4)}')

In [ ]:
import numpy as np
from scipy.linalg import expm, logm, sqrtm

# symmetric PD matrix
A = np.array([[4.0, 2.0, 1.0],
              [2.0, 3.0, 1.0],
              [1.0, 1.0, 2.0]])

# spectral decomposition: A = Q diag(lambda) Q^T
eigenvalues, Q = np.linalg.eigh(A)
print(f'eigenvalues: {eigenvalues.round(4)}')
reconstruction = Q @ np.diag(eigenvalues) @ Q.T
print(f'reconstruction error: {np.linalg.norm(A - reconstruction):.2e}')

# matrix square root: A^(1/2) = Q diag(sqrt(lambda)) Q^T
A_sqrt = Q @ np.diag(np.sqrt(eigenvalues)) @ Q.T
print(f'\nA_sqrt @ A_sqrt = A: {np.allclose(A_sqrt @ A_sqrt, A, atol=1e-8)}')
print(f'scipy sqrtm agrees:  {np.allclose(A_sqrt, sqrtm(A).real, atol=1e-8)}')

# matrix exponential: e^A = Q diag(e^lambda) Q^T
A_exp = Q @ np.diag(np.exp(eigenvalues)) @ Q.T
print(f'\nmatrix exponential (manual):\n{A_exp.round(4)}')
print(f'scipy expm agrees: {np.allclose(A_exp, expm(A), atol=1e-8)}')

In [ ]:
import numpy as np

# Moore-Penrose pseudoinverse via SVD
# A^+ = V diag(1/sigma_i for sigma_i > tol, else 0) U^T

A = np.array([[1.0, 2.0, 3.0],
              [4.0, 5.0, 6.0]])  # (2, 3) -- not square

A_pinv = np.linalg.pinv(A)
print(f'A shape    : {A.shape}')
print(f'A_pinv shape: {A_pinv.shape}')

# four Moore-Penrose conditions
print(f'\nMoore-Penrose conditions:')
print(f'A @ A+ @ A = A : {np.allclose(A @ A_pinv @ A, A)}')
print(f'A+ @ A @ A+ = A+: {np.allclose(A_pinv @ A @ A_pinv, A_pinv)}')
print(f'(A @ A+).T = A @ A+: {np.allclose((A @ A_pinv).T, A @ A_pinv)}')
print(f'(A+ @ A).T = A+ @ A: {np.allclose((A_pinv @ A).T, A_pinv @ A)}')

# pseudoinverse gives minimum-norm least-squares solution
b = np.array([1.0, 2.0])
x_min_norm = A_pinv @ b   # minimum ||x|| satisfying min ||Ax - b||
print(f'\nmin-norm least-squares x: {x_min_norm.round(4)}')
print(f'||x||: {np.linalg.norm(x_min_norm):.4f}')

In [ ]:
import numpy as np

# condition number: kappa = sigma_max / sigma_min
# high kappa = ill-conditioned = numerically unstable

def condition_number(A):
    S = np.linalg.svd(A, compute_uv=False)
    return S[0] / S[-1]

A_good = np.array([[2.0, 1.0],
                   [1.0, 2.0]])  # well-conditioned

A_bad  = np.array([[1.0, 1.0],
                   [1.0, 1.0001]])  # nearly singular

print(f'well-conditioned kappa : {condition_number(A_good):.2f}')
print(f'ill-conditioned kappa  : {condition_number(A_bad):.2e}')

# demonstrate: small input perturbation causes large output change when ill-conditioned
b = np.array([2.0, 2.0])
db = np.array([1e-4, 0.0])   # tiny perturbation

x      = np.linalg.solve(A_bad, b)
x_pert = np.linalg.solve(A_bad, b + db)
print(f'\ninput perturbation   ||db||   : {np.linalg.norm(db):.1e}')
print(f'output perturbation  ||dx||   : {np.linalg.norm(x_pert - x):.1e}')
print(f'amplification ratio           : {np.linalg.norm(x_pert - x) / np.linalg.norm(db):.1e}')

In [ ]:
import torch

A = torch.tensor([[4.0, 2.0, 1.0],
                  [2.0, 3.0, 1.0],
                  [1.0, 1.0, 2.0]])

# eigendecomposition of symmetric matrix
eigenvalues, eigenvectors = torch.linalg.eigh(A)
print(f'eigenvalues : {eigenvalues}')

# spectral reconstruction
A_recon = eigenvectors @ torch.diag(eigenvalues) @ eigenvectors.T
print(f'reconstruction match: {torch.allclose(A, A_recon, atol=1e-5)}')

# condition number
S = torch.linalg.svdvals(A)
kappa = S[0] / S[-1]
print(f'condition number: {kappa.item():.4f}')

In [ ]:
import tensorflow as tf
import numpy as np

A = tf.constant([[4.0, 2.0, 1.0],
                 [2.0, 3.0, 1.0],
                 [1.0, 1.0, 2.0]])

eigenvalues, eigenvectors = tf.linalg.eigh(A)
print(f'TF eigenvalues: {eigenvalues.numpy().round(4)}')

A_recon = eigenvectors @ tf.linalg.diag(eigenvalues) @ tf.transpose(eigenvectors)
print(f'reconstruction match: {np.allclose(A_recon.numpy(), A.numpy(), atol=1e-5)}')

# SVD for condition number
S = tf.linalg.svd(A, compute_uv=False)
kappa = S[0] / S[-1]
print(f'condition number: {kappa.numpy():.4f}')

In [ ]:
import numpy as np

# ML example: ZCA whitening using matrix square root of covariance
# ZCA whitening: X_white = X_centered @ C^(-1/2)
# Unlike PCA whitening it keeps the data in the original feature space orientation

np.random.seed(0)
cov_true = np.array([[4.0, 3.0],
                     [3.0, 4.0]])
X = np.random.multivariate_normal([0.0, 0.0], cov_true, size=300)

C = (1 / (X.shape[0] - 1)) * X.T @ X

# matrix inverse square root via eigendecomposition
eigenvalues, Q = np.linalg.eigh(C)
C_inv_sqrt = Q @ np.diag(1.0 / np.sqrt(eigenvalues)) @ Q.T

X_zca = X @ C_inv_sqrt

C_white = (1 / (X.shape[0] - 1)) * X_zca.T @ X_zca
print(f'original covariance:\n{C.round(4)}')
print(f'\nZCA whitened covariance (should be ~I):\n{C_white.round(4)}')
print(f'is identity: {np.allclose(C_white, np.eye(2), atol=1e-2)}')